# JavaScript 进阶 - 第16天

本天学习浅拷贝与深拷贝、异常处理、函数 `this`、`call/apply/bind`，并用防抖和节流优化高频事件。

## 一、浅拷贝与深拷贝

浅拷贝只复制第一层。基本类型复制值，数组和对象等引用类型复制地址，因此修改嵌套对象会影响原对象。常见写法有展开运算符和 `Object.assign`。

```javascript
const copy = { ...obj }
const copy2 = Object.assign({}, obj)
```

课堂示例：

```html
<!DOCTYPE html>
<html lang="zh-CN">

<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>Document</title>
</head>

<body>
  <script>
    const obj = {
      name: 'wangwu',
      age: 18,
      sex: '男',
      family: {
        father: '张三',
        mother: '李四'
      }
    }
    // 浅拷贝和深拷贝只针对引用类型（数组、对象）
    // 浅拷贝拷贝的是对象的地址，深拷贝拷贝的是对象的内容

    // 浅拷贝
    // 简单数据类型直接拷贝值，复杂数据类型拷贝地址（引用类型：数组、对象）
    // const o = { ...obj }
    // console.log(o)
    // o.age = 20
    // o.family.father = '王三'
    // console.log(o)
    // console.log(obj)

    const o = {}
    Object.assign(o, obj)
    o.age = 20
    o.family.father = '王三'
    console.log(o)
    console.log(obj)

  </script>
</body>

</html>
```

深拷贝会递归复制所有嵌套数据，使新旧对象互不影响。课堂演示了手写递归、Lodash 的 `_.cloneDeep` 和 JSON 序列化三种方式。

手写递归时先判断数组，再判断普通对象，因为数组也属于对象；遇到基本类型直接赋值。

```html
<!DOCTYPE html>
<html lang="zh-CN">

<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>Document</title>
</head>

<body>
  <script>
    // 深拷贝
    // 1. 递归拷贝
    // 2. loadsh
    // 3. JSON.stringify() 把对象转换为字符串，再用 JSON.parse() 把字符串转换为对象

    // 递归拷贝
    const obj = {
      uname: '张三',
      age: 18,
      hobby: ['篮球', '足球'],
      family: {
        name: '李四',
        age: 40
      }
    }
    const o = {}
    // 拷贝函数
    function deepCopy(newObj, oldObj) {
      for (let k in oldObj) {
        // 处理数组等复杂数据类型的问题
        // k 属性名 oldObj[k] 属性值
        if (oldObj[k] instanceof Array) {
          newObj[k] = []
          // 递归拷贝
          deepCopy(newObj[k], oldObj[k])
        } else if (oldObj[k] instanceof Object) {
          // 因为数组也是对象，所以必须先判断 数组，再判断 对象
          newObj[k] = {}
          // 递归拷贝
          deepCopy(newObj[k], oldObj[k])
        } else {
          newObj[k] = oldObj[k]
        }
      }
    }
    deepCopy(o, obj)
    console.log(o)
    o.age = 20
    o.hobby[0] = '跑步'
    o.family.age = 44
    console.log(obj)
  </script>
</body>

</html>
```

Lodash 方式：

```html
<!DOCTYPE html>
<html lang="zh-CN">

<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>Document</title>
</head>

<body>
  <script src="./lodash.min.js"></script>
  <script>
    const obj = {
      uname: '张三',
      age: 18,
      hobby: ['篮球', '足球'],
      family: {
        name: '李四',
        age: 40
      }
    }
    const o = _.cloneDeep(obj)
    console.log(o)
    o.family.age = 44
    console.log(obj)
  </script>
</body>

</html>
```

JSON 方式简单但有限制：函数、`undefined`、`Symbol` 会丢失，`Date` 等特殊对象也会被转换，存在循环引用时会报错。

```html
<!DOCTYPE html>
<html lang="zh-CN">

<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>Document</title>
</head>

<body>
  <script>
    const obj = {
      uname: '张三',
      age: 18,
      hobby: ['篮球', '足球'],
      family: {
        name: '李四',
        age: 40
      }
    }
    // // 把对象转换为 JSON 字符串
    // const str = JSON.stringify(obj)
    // console.log(str)
    // // 把字符串转换为对象
    // const o = JSON.parse(str)
    // console.log(o)

    // 3. JSON.stringify() 把对象转换为字符串，再用 JSON.parse() 把字符串转换为对象
    const o = JSON.parse(JSON.stringify(obj))
    console.log(o)
    o.age = 20
    o.hobby[0] = '跑步'
    o.family.age = 44
    console.log(obj)
  </script>
</body>

</html>
```

## 二、定时器与异常处理

`setTimeout` 只执行一次，递归调用可以模拟 `setInterval`。递归定时器便于在每次执行后决定是否继续，也能避免任务堆积。

```html
<!DOCTYPE html>
<html lang="zh-CN">

<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>Document</title>
</head>

<body>
  <div></div>
  <script>
    function getTime() {
      document.querySelector('div').innerHTML = new Date().toLocaleString()
      // 模拟setInterval效果
      setTimeout(getTime, 1000)
    }
    getTime()
  </script>
</body>

</html>
```

`throw` 主动抛出异常，抛出后当前流程停止。通常抛出 `new Error('错误信息')`，便于保留错误类型和 `message`。

```html
<!DOCTYPE html>
<html lang="zh-CN">

<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>Document</title>
</head>

<body>
  <script>
    // 1. throw 抛出异常信息，程序会停止运行
    // 2. throw 后面跟的是错误提示信息
    // 3. Error 对象配合 throw 使用，能够设置更详细的错误信息
    function fn(x, y) {
      if (!x || !y) {
        // throw '参数不能为空'
        throw new Error('参数不能为空')
      }
      return x + y
    }
    console.log(fn())
  </script>
</body>

</html>
```

`try...catch...finally` 用于捕获异常：可能出错的代码放入 `try`，错误信息在 `catch` 中处理，`finally` 无论是否出错都会执行。`catch` 处理错误后程序可以继续；需要中断时应显式 `return` 或再次 `throw`。

```html
<!DOCTYPE html>
<html lang="zh-CN">

<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>Document</title>
</head>

<body>
  <p>123</p>
  <script>
    function fn() {
      try {
        // 可能发生错误的代码 要写到 try 代码块中
        const p = document.querySelector('.p')
        p.style.color = 'red'
      } catch (err) {
        // 发生错误后，会执行 catch 代码块拦截错误，但是不中断程序运行
        console.log(err.message) // message 是浏览器内置的属性，用来获取错误提示信息
        // 需要添加 return 语句中断程序
        // return
        throw new Error('查询元素失败')
      } finally {
        // 无论是否发生错误，finally 代码块中的代码都会执行
        alert('程序结束')
      }
      // console.log('程序继续运行')

    }
    fn()
  </script>
</body>

</html>
```

## 三、普通函数的 this

普通函数的 `this` 由调用方式决定：顶层普通函数调用通常指向 `window`，对象方法调用指向调用对象，DOM 事件处理函数中的 `this` 指向当前元素，定时器回调中的普通函数通常指向 `window`。箭头函数沿用定义位置外层的 `this`。

```html
<!DOCTYPE html>
<html lang="zh-CN">

<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>Document</title>
</head>

<body>
  <button>点击</button>
  <script>
    console.log(this) // window
    function fn() {
      console.log(this) // window
    }
    window.fn()
    setTimeout(() => {
      console.log(this) // window
    }, 1000)
    document.querySelector('button').addEventListener('click', function () {
      console.log(this) // 点击的按钮
    })

    // 箭头函数的this指向见 day1 - 11-箭头函数的this指向.html
  </script>
</body>

</html>
```

## 四、改变函数 this

### 4.1 `call`

`fn.call(thisArg, arg1, arg2)` 会立即调用函数，并把 `this` 改为 `thisArg`，参数逐个传入。

```html
<!DOCTYPE html>
<html lang="zh-CN">

<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>Document</title>
</head>

<body>
  <script>
    const obj = {
      name: '张三',
      age: 18,
      sex: '男'
    }
    function fn(x, y) {
      console.log(this)
      console.log(x + y)
    }
    // call 方法
    // 1. 调用函数
    // 2. 改变函数的this指向
    // function.call(thisArg, arg1, arg2, ...) 
    // 1.1 thisArg: 改变后的this指向
    // 1.2 arg1, arg2, ...: 传递的参数
    fn.call(obj, 10, 20)
  </script>
</body>

</html>
```

### 4.2 `apply`

`fn.apply(thisArg, [args])` 同样立即调用并改变 `this`，参数必须放在数组中。适合把数组作为参数传给函数，例如 `Math.max.apply(null, arr)`。

```html
<!DOCTYPE html>
<html lang="zh-CN">

<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>Document</title>
</head>

<body>
  <script>
    const obj = {
      name: '张三',
      age: 18,
      sex: '男'
    }
    function fn(x, y) {
      console.log(this)
      console.log(x + y)
    }
    // apply 方法
    // 1. 调用函数
    // 2. 改变函数的this指向
    // function.apply(thisArg, [argsArray])
    // 1.1 thisArg: 改变后的this指向
    // 1.2 argsArray: 传递的参数数组
    fn.apply(obj, [10, 20])
    // call 方法和 apply 方法的区别
    // 1. call 方法传递的参数是逐个传递的
    // 2. apply 方法传递的参数是一个数组

    // 使用场景：求数组的最大值 
    // const max = Math.max(1, 2, 3, 4, 5)
    const max = Math.max.apply(null, [1, 2, 3, 4, 5])
    console.log(max)

  </script>
</body>

</html>
```

### 4.3 `bind`

`bind` 不会立即调用函数，而是返回一个新的绑定函数。它可以预先固定 `this` 和部分参数，适合延迟执行、事件回调或定时器回调。

```html
<!DOCTYPE html>
<html lang="zh-CN">

<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>Document</title>
</head>

<body>
  <button>点击</button>
  <script>
    const obj = {
      name: '张三',
      age: 18,
      sex: '男'
    }
    function fn(x, y) {
      console.log(this)
      console.log(x + y)
    }
    // 1. bind 不会调用函数
    // 2. bind 能改变函数的this指向
    // 3. 返回值是一个新的函数，这个函数的this指向是 更改后的this指向
    const newFn = fn.bind(obj, 10, 20)
    newFn()

    // 使用场景：有一个按钮，点击里面就禁用，2秒后自动开启
    document.querySelector('button').addEventListener('click', function () {
      this.disabled = true // this 指向btn按钮
      window.setTimeout(function () {
        // 这里的this指向的是window对象
        // 我们需要改变this指向，指向btn按钮
        // 可以使用bind方法
        this.disabled = false
      }.bind(this), 2000) // 这里的this指向的是btn按钮
    })
  </script>
</body>

</html>
```

`call` 和 `apply` 的区别在参数传递形式；`bind` 的区别在于返回新函数且不会立即执行。

## 五、防抖

防抖的核心是：事件连续触发时不断重新计时，只有停止触发达到延迟时间后，才执行最后一次。适合搜索框输入检测、窗口 resize 后计算布局等场景。

实现时先清除旧定时器，再创建新定时器。Lodash 提供了 `_.debounce(fn, delay)`。

```html
<!DOCTYPE html>
<html lang="zh-CN">

<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>Document</title>
  <style>
    .box {
      width: 200px;
      height: 200px;
      background-color: pink;
      text-align: center;
      line-height: 200px;
      color: #fff;
      font-size: 30px;
    }
  </style>
</head>

<body>
  <div class="box"></div>
  <script src="./lodash.min.js"></script>
  <script>
    // 防抖核心思想是：当事件被触发时，并不立即执行函数，而是等待一个特定的延迟时间，
    // 如果在这个延迟时间内事件又被触发了，则重新开始计时，直到延迟时间结束后才真正执行函数

    // 场景：鼠标在盒子上移动。里面的数字就会 +1
    const box = document.querySelector('.box')
    let num = 0
    // box.addEventListener('mousemove', function () {
    //   // 如果里面有大量消耗性能的代码，比如dom操作、数据处理，容易造成卡顿
    //   num++
    //   this.innerHTML = num
    // })

    // 1. 利用 loadsh 库的 debounce 方法实现防抖
    // 第一个参数是要执行的函数
    // 第二个参数是延迟时间，单位是毫秒
    // 事件触发后，等待500毫秒后执行函数
    // box.addEventListener('mousemove', _.debounce(function () {
    //   num++
    //   this.innerHTML = num
    // }, 500))

    // 2. 利用 setTimeout 实现防抖
    // 事件触发后，等待500毫秒后执行函数
    let timer = null
    box.addEventListener('mousemove', function () {
      // 先清除之前的定时器
      clearTimeout(timer)
      // 重新设置定时器
      timer = setTimeout(() => {
        num++
        this.innerHTML = num
      }, 500)
    })
  </script>
</body>

</html>
```

## 六、节流

节流的核心是：在固定时间窗口内无论触发多少次，只执行一次。适合 `mousemove`、`scroll`、`resize` 等高频事件。

手写节流可以用定时器作为锁：第一次触发创建定时器，窗口结束后执行函数并把定时器置空，下一窗口才能再次执行。

```javascript
function throttle(fn, delay) {
  let timer = null
  return function () {
    if (!timer) {
      timer = setTimeout(() => {
        fn()
        timer = null
      }, delay)
    }
  }
}
```

```html
<!DOCTYPE html>
<html lang="zh-CN">

<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>Document</title>
  <style>
    .box {
      width: 200px;
      height: 200px;
      background-color: pink;
      text-align: center;
      line-height: 200px;
      color: #fff;
      font-size: 30px;
    }
  </style>
</head>

<body>
  <div class="box"></div>
  <script src="./lodash.min.js"></script>
  <script>
    // 节流核心思想是：在一定时间内，无论触发多少次，只执行一次函数

    // 场景：鼠标在盒子上移动。里面的数字就会 +1
    const box = document.querySelector('.box')
    let num = 0
    // box.addEventListener('mousemove', function () {
    //   // 如果里面有大量消耗性能的代码，比如dom操作、数据处理，容易造成卡顿
    //   num++
    //   this.innerHTML = num
    // })

    // 1. 利用 loadsh 库的 throttle 方法实现节流
    // 第一个参数是要执行的函数
    // 第二个参数是延迟时间，单位是毫秒
    // 事件触发后，等待1000毫秒后执行函数
    // box.addEventListener('mousemove', _.throttle(function () {
    //   num++
    //   this.innerHTML = num
    // }, 1000))

    // 2. 利用 setTimeout 实现节流
    function throttle(fn, delay) {
      let timer = null
      return function () {
        if (!timer) {
          timer = setTimeout(function () {
            fn()
            // 在setTimeout函数中是无法删除timer的，因为timer还在运作
            // 所以需要在setTimeout函数中手动删除timer
            timer = null // 手动删除timer
          }, delay)
        }
      }
    }

    // 事件触发后，等待1000毫秒后执行函数
    box.addEventListener('mousemove', throttle(function () {
      num++
      console.log(num)
      // 不能用 this 因为此时的this指向的是window对象（setTimeout函数）
      box.innerHTML = num
    }, 1000))

    // 防抖：延迟单位时间后执行，单位时间内只执行最后一次
    // 常用于搜索框、手机号、邮箱验证输入检测

    // 节流：单位时间内只执行第一次
    // 常用于高频事件：鼠标移动mousemove、滚动条滚动scroll、窗口缩放resize等
  </script>
</body>

</html>
```

| 方式 | 执行时机 | 常见场景 |
| --- | --- | --- |
| 防抖 | 连续触发停止后执行最后一次 | 搜索输入、表单校验 |
| 节流 | 固定时间内最多执行一次 | 滚动、鼠标移动、窗口缩放 |

## 七、综合案例：视频播放进度记忆

视频的 `ontimeupdate` 触发频率较高，使用 Lodash 节流后每秒保存一次播放时间到 `localStorage`。视频加载完成时读取保存的时间并恢复播放位置。

```html
<!DOCTYPE html>
<html lang="en">

<head>
  <meta charset="UTF-8" />
  <meta http-equiv="X-UA-Compatible" content="IE=edge" />
  <meta name="viewport" content="width=device-width, initial-scale=1.0" />
  <meta name="referrer" content="never" />
  <title>综合案例</title>
  <style>
    * {
      padding: 0;
      margin: 0;
      box-sizing: border-box;
    }

    .container {
      width: 1200px;
      margin: 0 auto;
    }

    .video video {
      width: 100%;
      padding: 20px 0;
    }

    .elevator {
      position: fixed;
      top: 280px;
      right: 20px;
      z-index: 999;
      background: #fff;
      border: 1px solid #e4e4e4;
      width: 60px;
    }

    .elevator a {
      display: block;
      padding: 10px;
      text-decoration: none;
      text-align: center;
      color: #999;
    }

    .elevator a.active {
      color: #1286ff;
    }

    .outline {
      padding-bottom: 300px;
    }
  </style>
</head>

<body>
  <div class="container">
    <div class="header">
      <a href="http://pip.itcast.cn">
        <img src="https://pip.itcast.cn/img/logo_v3.29b9ba72.png" alt="" />
      </a>
    </div>
    <div class="video">
      <video src="https://v.itheima.net/LapADhV6.mp4" controls></video>
    </div>
    <div class="elevator">
      <a href="javascript:;" data-ref="video">视频介绍</a>
      <a href="javascript:;" data-ref="intro">课程简介</a>
      <a href="javascript:;" data-ref="outline">评论列表</a>
    </div>
  </div>
  <script src="./lodash.min.js"></script>
  <script>
    const video = document.querySelector('video')
    // ontimeupdate 事件：视频播放时间更新时触发
    // 节流，一秒内只执行一次
    video.ontimeupdate = _.throttle(function () {
      // console.log(video.currentTime)
      localStorage.setItem('currentTime', video.currentTime)
    }, 1000)

    // onloadeddata 事件：视频元数据加载完成（包括视频时长、视频尺寸等）时触发
    video.onloadeddata = function () {
      // 检查是否有之前保存的播放时间
      video.currentTime = localStorage.getItem('currentTime') || 0
    }
  </script>
</body>

</html>
```

案例要点：`localStorage` 保存的是字符串，赋给 `currentTime` 时会自动转换为数字；没有历史记录时使用 `0`。如果实际项目中要保存多个视频，应使用视频 id 作为存储键，避免相互覆盖。

## 八、易错点

1. 浅拷贝不会复制嵌套引用，深拷贝要根据数据类型选择方案。
2. JSON 深拷贝不能完整处理函数、日期、循环引用等数据。
3. `catch` 只能捕获同步 `try` 代码块中的异常，异步异常要在异步回调或 Promise 链中处理。
4. `call/apply` 立即调用，`bind` 返回新函数。
5. 防抖执行最后一次，节流限制时间窗口内的执行次数。
6. 使用定时器节流时要在回调结束后释放锁，否则后续事件不会执行。

建议分别用原生代码实现搜索框防抖和页面滚动节流，再用 Lodash 版本对照验证。

## 九、资料来源

来源：[百度网盘分享](https://pan.baidu.com/s/1MxueOHXGQtmE-ypRq1kELA?pwd=9987)，提取码 `9987`。对应目录为 `1. Pink最新版前端JavaScript/3.JS进阶/资料/JavaScript进阶第四天`，课堂代码参考 `04-code` 原始 HTML 示例。